# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a guided workflow for loading, exploring, and processing the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Install mlcroissant and other required libraries
!pip install mlcroissant --quiet
!pip install matplotlib --quiet
!pip install seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata as a single object
metadata = dataset.metadata
print("Dataset name:", metadata.name)
print("Description:", metadata.description)
print("Version:", metadata.version)
print("Published:", metadata.datePublished)
print("License:", metadata.license)
print("Identifier:", metadata.identifier)


## 2. Data Overview
Review available record sets, fields, and their IDs.

By default, Croissant datasets expose record sets, fields, and columns, each referenced by an `@id`.

In [ ]:
# List available record sets using their @id
record_sets = metadata.recordSet
if not record_sets:
    print("No record sets found in this dataset. Attempting to extract from schema JSON...")
    # Attempt to load record sets directly from the schema (if not loaded above)
    import requests
    resp = requests.get(croissant_url)
    croissant_schema = resp.json()
    # Extract record sets from dataset JSON-LD
    # Find nodes of type cr:RecordSet
    def get_record_sets_ld(json_ld):
        record_sets = []
        if isinstance(json_ld, dict):
            if '@type' in json_ld and (
                json_ld['@type'] == 'cr:RecordSet' or json_ld['@type'] == 'RecordSet'):
                record_sets.append(json_ld)
            for v in json_ld.values():
                record_sets.extend(get_record_sets_ld(v))
        elif isinstance(json_ld, list):
            for item in json_ld:
                record_sets.extend(get_record_sets_ld(item))
        return record_sets
    ld_record_sets = get_record_sets_ld(croissant_schema)
    record_sets = ld_record_sets

record_set_ids = []

if isinstance(record_sets, list):
    for rs in record_sets:
        if isinstance(rs, dict) and '@id' in rs:
            print(f"Record Set: {rs['@id']}, name: {rs.get('name','')} (type: {rs.get('@type','')})")
            record_set_ids.append(rs['@id'])
        elif isinstance(rs, str):
            print(f"Record Set: {rs}")
            record_set_ids.append(rs)
else:
    print("Found record sets of unrecognized format.")

# For each record set, list its fields and their @id
print("\nSummary of field @id for each Record Set:")
fields_by_record_set = {}

for rs_id in record_set_ids:
    fields = []
    for rs in record_sets:
        if (isinstance(rs, dict) and rs.get('@id') == rs_id):
            # Check for fields
            rs_fields = rs.get('field', [])
            if isinstance(rs_fields, list):
                for f in rs_fields:
                    if isinstance(f, dict):
                        fields.append(f['@id'])
                    elif isinstance(f, str):
                        fields.append(f)
            elif isinstance(rs_fields, str):
                fields.append(rs_fields)
    fields_by_record_set[rs_id] = fields
    print(f"- Record Set {rs_id}: fields: {fields}")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Use the record set and field `@id`s discovered above.

In [ ]:
# Extract records from each record set
dataframes = {}
for rs_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rs_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[rs_id] = df
            print(f"\nRecord Set: {rs_id} loaded: {df.shape[0]} records, fields: {df.columns.tolist()}")
        else:
            print(f"No records found for Record Set {rs_id}")
    except Exception as e:
        print(f"Failed to load records for {rs_id}: {e}")

# Show summary for first record set
if record_set_ids:
    primary_rs_id = record_set_ids[0]
    print("\nColumns in first loaded data frame:", dataframes[primary_rs_id].columns.tolist())
    display(dataframes[primary_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering records, normalizing numeric fields, and grouping data by key attributes.

You can use field `@id` from the previous overview to reference columns.

In [ ]:
# EDA for the primary record set
df = dataframes[primary_rs_id]
print(f"EDA will use record set {primary_rs_id}.")

# Try to identify a numeric column (by common field names or datatypes)
numeric_col_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col] )]
if not numeric_col_candidates:
    # If not detected, try by heuristics
    for col in df.columns:
        if ('age' in col.lower()) or ('interval' in col.lower()) or ('count' in col.lower()) or ('number' in col.lower()):
            try:
                df[col] = pd.to_numeric(df[col], errors='coerce')
                numeric_col_candidates.append(col)
            except Exception:
                pass
if not numeric_col_candidates:
    print("No numeric columns found for analysis.")

if numeric_col_candidates:
    numeric_field_id = numeric_col_candidates[0]
    print(f"Selected numeric field for analysis: {numeric_field_id}")

    # Filter for values above a threshold
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"\nFiltered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical field if present
    group_field_candidates = [col for col in df.columns if df[col].dtype=='object' and col!=numeric_field_id]
    if group_field_candidates:
        group_field_id = group_field_candidates[0]
        print(f"\nGrouping by field: {group_field_id}")
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(grouped_df.head())
else:
    print("No numeric columns available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset using matplotlib and seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_col_candidates and group_field_candidates:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field selected, show boxplot
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()


## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR^2 dataset contains clinical, pathological, anatomical, and molecular biomarker variables for cancer survivors with second primary colorectal cancer.
- Using `mlcroissant`, the dataset metadata, record sets, and fields were loaded and examined.
- Numeric fields, such as age or diagnosis interval, can be filtered and normalized for exploratory and statistical analysis.
- Grouping and visualizations by anatomical or clinical characteristics reveal sample distributions and potential clinical stratification trends.
- The notebook demonstrates how to reference data elements by `@id` and process Croissant datasets reproducibly.